In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Good default for your RTX 3080 Ti 16GB: 7B Instruct in 4-bit
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()


def generate_sentences(
    word: str,
    meaning: str = "",
    num_sentences: int = 3,
    min_chars: int = 10,
    max_chars: int = 20,
    script: str = "auto",  # "traditional" | "simplified" | "auto"
):
    system = (
        "You generate natural Mandarin example sentences for language learners. "
        "Follow constraints exactly. Output only the sentence text."
    )

    if script == "traditional":
        script_line = "Use Traditional Chinese characters."
    elif script == "simplified":
        script_line = "Use Simplified Chinese characters."
    else:
        script_line = "Use natural Chinese characters matching the input word."

    results = []
    for _ in range(num_sentences):
        user = (
            f"Target word: {word}\n"
            f"Meaning (English): {meaning}\n"
            f"Constraints:\n"
            f"- One Chinese sentence only\n"
            f"- Must include the target word exactly once\n"
            f"- {min_chars}–{max_chars} Chinese characters (excluding punctuation)\n"
            f"- Everyday context\n"
            f"- No extra text, no numbering\n"
            f"- {script_line}\n"
        )

        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]

        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only the newly generated tokens (avoid echoing prompt)
        new_tokens = out[0, inputs["input_ids"].shape[-1] :]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        # Light cleanup (model sometimes adds quotes)
        text = text.strip('"“”')
        results.append(text)

    return results


/home/maishuji/Workplace/chinese-flash-cards/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:10<00:00,  2.57s/it]


In [4]:
prompt = "可怕"
generated_sentences = generate_sentences(prompt, meaning="scary/terrifying", num_sentences=3, script="traditional")
for i, sentence in enumerate(generated_sentences, 1):
    print(f"Sentence {i}:\n{sentence}\n")


Sentence 1:
這場恐怖片真的太可怕了。

Sentence 2:
這部電影太可怕了，我不敢再看了。

Sentence 3:
這種声音好可怕。

